# Elasticity and mechanics: analytical and finite-difference solution

This notebook is designed for teaching materials science students.

We study **1D linear elasticity** in a bar. This is one of the most important
differential-equation models in mechanics.

We begin with the equilibrium equation

$$
\frac{d\sigma}{dx} + f(x) = 0,
$$

where

- $\sigma(x)$ is stress,
- $f(x)$ is body force per unit volume.

Using Hooke's law in 1D,

$$
\sigma = E \frac{du}{dx},
$$

we obtain the displacement equation

$$
\frac{d}{dx}\left(E \frac{du}{dx}\right) + f(x) = 0.
$$

For constant \(E\), this becomes

$$
E \frac{d^2 u}{dx^2} + f(x) = 0.
$$

We will:

1. derive the governing equation,
2. solve a simple case analytically,
3. solve the same problem by finite differences,
4. compare displacement, strain, and stress.

## 1. Imports and setup

In [ ]:
import sympy as sp
sp.init_printing()

import numpy as np
import matplotlib.pyplot as plt

## 2. Governing equation

Let $u(x)$ be the displacement field in a 1D bar of length $L$.

The equilibrium equation is

$$
\frac{d\sigma}{dx} + f(x) = 0.
$$

Using

$$
\sigma = E \frac{du}{dx},
$$

we get

$$
\frac{d}{dx}\left(E \frac{du}{dx}\right) + f(x) = 0.
$$

For this notebook, we first assume a constant elastic modulus \(E\), so the equation becomes

$$
E u''(x) + f(x) = 0.
$$

In [ ]:
x = sp.symbols('x', real=True)
E, L, f0 = sp.symbols('E L f0', positive=True, real=True)
u = sp.Function('u')

ode = sp.Eq(E * sp.diff(u(x), x, 2) + f0, 0)
ode

## 3. Example problem

We solve the bar problem on $0 \le x \le L$ with:

- constant body force $f(x)=f_0$,
- left end fixed: $u(0)=0$,
- right end traction free: $\sigma(L)=0$, so
  $$
  E u'(L)=0 \quad \Rightarrow \quad u'(L)=0.
  $$

So the boundary conditions are

$$
u(0)=0, \qquad u'(L)=0.
$$

## 4. Analytical solution with SymPy

In [ ]:
general_sol = sp.dsolve(ode)
general_sol

The general solution should contain two integration constants. We now apply the boundary conditions.

In [ ]:
C1, C2 = sp.symbols('C1 C2')
u_general = general_sol.rhs

eq1 = sp.Eq(u_general.subs(x, 0), 0)
eq2 = sp.Eq(sp.diff(u_general, x).subs(x, L), 0)

const_sol = sp.solve([eq1, eq2], [C1, C2], dict=True)[0]
const_sol

In [ ]:
u_exact = sp.simplify(u_general.subs(const_sol))
u_exact

So the analytical displacement field is

$$
u(x) = \frac{f_0}{E}\left(Lx - \frac{x^2}{2}\right).
$$

This is a parabola.

The strain is

$$
\varepsilon(x) = \frac{du}{dx},
$$

and the stress is

$$
\sigma(x) = E \varepsilon(x).
$$

In [ ]:
eps_exact = sp.simplify(sp.diff(u_exact, x))
sig_exact = sp.simplify(E * eps_exact)

u_exact, eps_exact, sig_exact

## 5. Verify the analytical solution

In [ ]:
check_ode = sp.simplify(E * sp.diff(u_exact, x, 2) + f0)
check_bc1 = sp.simplify(u_exact.subs(x, 0))
check_bc2 = sp.simplify(sp.diff(u_exact, x).subs(x, L))

check_ode, check_bc1, check_bc2

If the outputs are all zero, then the exact solution satisfies both the differential equation and the boundary conditions.

## 6. Physical interpretation

For the exact solution

$$
u(x)=\frac{f_0}{E}\left(Lx - \frac{x^2}{2}\right),
$$

we see that:

- displacement is largest near the free end,
- strain decreases linearly with $x$,
- stress also decreases linearly with $x$,
- stress becomes zero at $x=L$, consistent with the traction-free boundary condition.

In [ ]:
u_exact_func = sp.lambdify((x, E, L, f0), u_exact, 'numpy')
eps_exact_func = sp.lambdify((x, E, L, f0), eps_exact, 'numpy')
sig_exact_func = sp.lambdify((x, E, L, f0), sig_exact, 'numpy')

In [ ]:
E_val = 200.0
L_val = 1.0
f0_val = 10.0

xv = np.linspace(0, L_val, 300)

plt.plot(xv, u_exact_func(xv, E_val, L_val, f0_val))
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Analytical displacement field")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(xv, eps_exact_func(xv, E_val, L_val, f0_val), label="strain")
# plt.plot(xv, sig_exact_func(xv, E_val, L_val, f0_val), label="stress")
plt.xlabel("x")
plt.ylabel("value")
plt.title("Analytical strain and stress")
plt.legend()
plt.show()

## 7. Finite-difference method

For constant $E$, the governing equation is

$$
E u''(x) + f_0 = 0.
$$

We approximate the second derivative at interior node $i$ by

$$
u''(x_i) \approx \frac{u_{i+1} - 2u_i + u_{i-1}}{\Delta x^2}.
$$

So the finite-difference equation becomes

$$
E \frac{u_{i+1} - 2u_i + u_{i-1}}{\Delta x^2} + f_0 = 0.
$$

Rearranging:

$$
u_{i-1} - 2u_i + u_{i+1} = -\frac{f_0 \Delta x^2}{E}.
$$

For the boundary conditions:

### Left boundary
$$
u(0)=0
$$

so we set

$$
u_0 = 0.
$$

### Right boundary
$$
u'(L)=0
$$

We approximate the derivative by a backward difference:

$$
\frac{u_N - u_{N-1}}{\Delta x} = 0
\quad \Rightarrow \quad
u_N = u_{N-1}.
$$

In [ ]:
def elasticity_fd_constant_E(E, L, f0, N):
    x = np.linspace(0, L, N+1)
    dx = x[1] - x[0]

    A = np.zeros((N+1, N+1))
    b = np.zeros(N+1)

    # Left boundary: u(0) = 0
    A[0, 0] = 1.0
    b[0] = 0.0

    # Interior nodes
    for i in range(1, N):
        A[i, i-1] = 1.0
        A[i, i]   = -2.0
        A[i, i+1] = 1.0
        b[i] = -(f0 * dx**2) / E

    # Right boundary: u_N - u_{N-1} = 0
    A[N, N] = 1.0
    A[N, N-1] = -1.0
    b[N] = 0.0

    u = np.linalg.solve(A, b)
    return x, u

## 8. Compare analytical and finite-difference displacement

In [ ]:
E_val = 200.0
L_val = 1.0
f0_val = 10.0
N = 40

x_fd, u_fd = elasticity_fd_constant_E(E_val, L_val, f0_val, N)

xv = np.linspace(0, L_val, 400)
u_ex = u_exact_func(xv, E_val, L_val, f0_val)

plt.plot(xv, u_ex, 'k-', lw=2, label='exact')
plt.plot(x_fd, u_fd, 'o', ms=4, label='finite difference')
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Exact vs finite-difference displacement")
plt.legend()
plt.show()

print("Maximum absolute displacement error =",
      np.max(np.abs(u_fd - u_exact_func(x_fd, E_val, L_val, f0_val))))

## 9. Compute strain and stress from the finite-difference solution

We estimate strain from the displacement field.

At interior points we use a central difference:

$$
\varepsilon_i \approx \frac{u_{i+1}-u_{i-1}}{2\Delta x}.
$$

At the boundaries we use one-sided differences.

In [ ]:
def strain_from_displacement(x, u):
    dx = x[1] - x[0]
    eps = np.zeros_like(u)

    eps[0] = (u[1] - u[0]) / dx
    eps[-1] = (u[-1] - u[-2]) / dx
    eps[1:-1] = (u[2:] - u[:-2]) / (2*dx)

    return eps

In [ ]:
eps_fd = strain_from_displacement(x_fd, u_fd)
sig_fd = E_val * eps_fd

eps_ex_fdgrid = eps_exact_func(x_fd, E_val, L_val, f0_val)
sig_ex_fdgrid = sig_exact_func(x_fd, E_val, L_val, f0_val)

plt.plot(x_fd, eps_ex_fdgrid, 'k-', lw=2, label='exact strain')
plt.plot(x_fd, eps_fd, 'o', ms=4, label='FD strain')
plt.xlabel("x")
plt.ylabel("strain")
plt.title("Exact vs finite-difference strain")
plt.legend()
plt.show()

In [ ]:
plt.plot(x_fd, sig_ex_fdgrid, 'k-', lw=2, label='exact stress')
plt.plot(x_fd, sig_fd, 'o', ms=4, label='FD stress')
plt.xlabel("x")
plt.ylabel("stress")
plt.title("Exact vs finite-difference stress")
plt.legend()
plt.show()

## 10. Effect of mesh size

As the mesh is refined, the finite-difference solution should converge to the exact solution.

In [ ]:
E_val = 200.0
L_val = 1.0
f0_val = 10.0

N_list = [10, 20, 40, 80]
errors = []

for N in N_list:
    x_fd, u_fd = elasticity_fd_constant_E(E_val, L_val, f0_val, N)
    u_ex = u_exact_func(x_fd, E_val, L_val, f0_val)
    err = np.max(np.abs(u_fd - u_ex))
    errors.append(err)

plt.plot(N_list, errors, 'o-')
plt.xlabel("N")
plt.ylabel("max displacement error")
plt.title("Mesh refinement study")
plt.show()

for N, err in zip(N_list, errors):
    print(f"N = {N:3d}, max error = {err:.6e}")

## 11. Variable modulus form (conceptual extension)

In real materials problems, \(E\) may vary with position. Then the governing equation is

\[
\frac{d}{dx}\left(E(x) \frac{du}{dx}\right) + f(x) = 0.
\]

This is more general than

\[
E u'' + f = 0.
\]

A finite-difference method can still be built, but now it is better to discretize the flux-like quantity

\[
E(x) \frac{du}{dx}
\]

at cell faces.

This idea is closely related to finite-volume and finite-element methods.

## 12. Another common example: end load instead of body force

Another standard mechanics problem is

\[
\frac{d\sigma}{dx}=0
\]

with

- \(u(0)=0\),
- \(\sigma(L)=\sigma_0\).

Then stress is constant, strain is constant, and displacement is linear:

\[
u(x)=\frac{\sigma_0}{E}x.
\]

This is a good contrast with the curved displacement field caused by a body force.

## 13. Summary

In this notebook, we:

- started from the 1D equilibrium equation,
- used Hooke's law to obtain the displacement equation,
- solved a body-force problem analytically,
- solved the same problem using a finite-difference method,
- compared displacement, strain, and stress,
- studied convergence under mesh refinement.

This is foundational material for elasticity, solid mechanics, and numerical methods in materials science.

## 14. Optional exercises

1. Replace the constant body force with \(f(x)=f_0 x\) and solve analytically.
2. Change the right boundary condition from traction-free to a prescribed displacement.
3. Implement the variable-modulus equation \(\frac{d}{dx}(E(x)u')+f(x)=0\).
4. Compare this finite-difference formulation with a simple finite-element formulation.
5. Interpret the stress distribution physically in terms of force balance.